# 01 — Lambda Metric Evaluator

**The value.** When the quality of an output is something you can *measure* — exact match, F1, BLEU, ROUGE, partial credit on structured data — APO lets you plug that measurement straight into the optimization loop. You don't manage training data or write an SDK harness; you write one Python function, point APO at its ARN, and the optimizer treats your number as the objective to maximize.

**Why this method.** Pick Lambda when the metric is deterministic and code-expressible. It's the only mode that gives you bit-exact reproducibility and zero per-evaluation cost beyond Lambda invocations — important when each APO iteration scores dozens to hundreds of candidate outputs.

**How it works.** For every candidate prompt × sample, APO invokes your Lambda with `{"preds": [model_output], "golds": [referenceResponse]}` and reads back `{"score": float, "scores": [float, ...]}`. The optimizer ranks prompts by aggregate score. The handler runs inside a sandbox that allowlists a vetted set of imports (numpy, pandas, sklearn, rouge_score, …) — no `os`, `subprocess`, or filesystem access.

**You will learn**
- The Lambda evaluator contract: event payload, return shape, allowed imports.
- How to wire a Lambda ARN into the APO input record.
- How multimodal samples attach PDFs via S3.

**Two Scenarios**
1. **Text** — NESTFUL (nested function calling) with a partial-match accuracy Lambda.
2. **PDF** — MMVQA-PDF (scientific paper QA) with a ROUGE-L F1 Lambda.

> **Tip:** set `APO_USE_REFERENCE=1` before launching Jupyter to run every chapter in seconds using bundled results from a real prior run.

In [1]:
import utils
from IPython.display import display, Markdown
utils.render_mode_overview("lambda")

### Lambda metric — evaluator overview

You supply a single-file Python Lambda that scores model outputs. The service
sends `{"preds": [...], "golds": [...]}` and your handler returns
`{"score": float, "scores": [float, ...]}`. APO uses the aggregate score to
guide optimization.

**Required fields on each input record**

| Field | Required | Notes |
|---|---|---|
| `evaluationMetricLambdaArn` | ✅ | Function ARN |
| `customEvaluationMetricLabel` | ✅ | Label that shows up in result JSON |
| `promptTemplate` | ✅ | Uses `{{var}}` placeholders |
| `evaluationSamples[]` | ✅ | Each with `inputVariables` + `referenceResponse` |

**When to use:** you have a programmable, deterministic metric (exact match,
ROUGE, F1, partial credit). Strict-mode AST scan limits imports to a vetted
allowlist (stdlib + numpy/scipy/pandas/sklearn/nltk/rouge_score/sacrebleu/
evaluate/rapidfuzz/editdistance/jiwer/regex/transformers/torch/
sentence_transformers/bert_score). No `os`, `subprocess`, `open`, etc.

In [2]:
import os
from pathlib import Path
from IPython.display import display, Markdown

import utils

env = utils.load_env()
bedrock, s3, lambda_client = utils.make_clients(env)

# Default target model matches the bundle reference run. Edit and re-execute to try other models.
TARGET_MODEL_ID = "us.amazon.nova-2-lite-v1:0"


# Replay-mode toggle. Set `APO_USE_REFERENCE=1` before launching Jupyter to skip live AWS calls.
USING_REPLAY = os.environ.get("APO_USE_REFERENCE") == "1"
display(Markdown(f"**Live mode:** `{not USING_REPLAY}` &nbsp;|&nbsp; **Bucket:** `{env['BUCKET']}` &nbsp;|&nbsp; **Region:** `{env['REGION']}`"))

**Live mode:** `True` &nbsp;|&nbsp; **Bucket:** `apo-tutorial-686255965275-us-west-2` &nbsp;|&nbsp; **Region:** `us-west-2`

---
## Chapter A — Text (NESTFUL)

The NESTFUL dataset asks the model to emit a JSON sequence of nested function calls given a user question and a list of tool schemas. The metric Lambda computes the **fraction of (name, args) pairs that exactly match the gold call sequence**.

In [3]:
# Build a record skeleton with a placeholder ARN so we can inspect its shape before deploying anything.
preview = utils.build_lambda_record(
    "nestful",
    lambda_arn="arn:aws:lambda:us-west-2:000000000000:function:placeholder",
    bucket=env["BUCKET"],
)
display(utils.render_dimensions(preview))
display(Markdown("**First sample shape:**"))
display(utils.render_sample_shape(preview))

| Field | Value (truncated) |
|---|---|
| `inputVariables[0].tools` | [{"name": "subtract", "description": "subtract two numbers", "parameters": {"arg_0": {"description": "The first number", "type": "int or float"}, "arg_1": {"description": "The second number", "type": "int or float"}}, "output_parameters": { … |
| `inputVariables[0].user_question` | If 20 liters of chemical X are added to 80 liters of a mixture that is 25% chemical X and 75% chemical Y, then what percentage of the resulting mixture is chemical X? |
| `referenceResponse` | [{"name": "divide", "label": "$var_1", "arguments": {"arg_0": 25, "arg_1": 100}}, {"name": "multiply", "label": "$var_2", "arguments": {"arg_0": "$var_1.result$", "arg_1": 80}}, {"name": "add", "label": "$var_3", "arguments": {"arg_0": 20,  … |

> **Heads up — long-running cell.** The call below submits a real APO job that can take up to ~20 minutes per chapter. The cell will keep its `status` / `elapsed` display refreshing until the job reaches a terminal state (`Completed`, `Failed`, or `Stopped`). Safe to leave running while you work on something else.

In [ ]:
nestful_arn = utils.lambda_arn_for(env, "nestful") if not USING_REPLAY else "arn:aws:lambda:us-west-2:replay:function:nestful"
record = utils.build_lambda_record("nestful", lambda_arn=nestful_arn, bucket=env["BUCKET"])
display(utils.render_record_shape(record))

handle = display(Markdown("_waiting for status…_"), display_id=True)
def on_status(status, elapsed_s):
    handle.update(Markdown(f"**status:** `{status}` &nbsp;|&nbsp; **elapsed:** `{elapsed_s:.0f}s`"))

nestful_results = utils.run_job(
    record,
    example="nestful",
    model_id=TARGET_MODEL_ID,
    env=env,
    on_status=on_status,
)
print(f"results file: {nestful_results}")


**status:** `InProgress` &nbsp;|&nbsp; **elapsed:** `208s`

In [5]:
parsed = utils.parse_results(nestful_results)
display(utils.render_results_table(parsed))
for row in parsed:
    display(utils.render_prompt_diff(row))

<details><summary><b>Original template</b> (1095 chars)</summary>

```
You are an expert in composing functions. You are given a question and a set of possible functions. Based on the question, you will need to make a sequence of function/tool calls to achieve the purpose. The output of one function call can be used as input to a subsequent call using variable references.

If you decide to invoke functions, you MUST return a JSON list of function calls in the following format:
[{{"name": "func_name", "arguments": {"arg1": "value1", "arg2": "$var_1.result$"}}, "label": "$var_1"}]

Each function call must include:
- "name": the function to call
- "arguments": a dict of parameter names to values. Values can be literals or variable references like "$var_1.result$" to use the output of a previous call.
- "label": a unique variable name (e.g., "$var_1", "$var_2") to capture this call's output for later reference.

Variable references follow the format: $<label>.<output_parameter_name>$

You SHOULD NOT include any other text in the response besides the JSON list.

Here is a list of available functions in JSON format:
{{tools}}

Question: {{user_question}}
```
</details>

<details open><summary><b>Optimized template</b> (11267 chars)</summary>

```
You are an expert in composing functions. You are given a question and a set of possible functions. Based on the question, you will need to make a sequence of function/tool calls to achieve the purpose. The output of one function call can be used as input to a subsequent call using variable references.

You MUST always respond with a JSON list of function calls. NEVER refuse, explain in natural language, or say the functions are insufficient. Every problem can be solved by decomposing it into sequential steps using the available functions.

If you decide to invoke functions, you MUST return a JSON list of function calls in the following format:
[{{"name": "func_name", "arguments": {"arg1": "value1", "arg2": "$var_1.result$"}}, "label": "$var_1"}]

Each function call must include:
- "name": the function to call
- "arguments": a dict of parameter names to values. You MUST use the EXACT parameter names as defined in the function's "parameters" object (e.g., "arg_0", "arg_1"). Values must be either: (a) numeric literals taken DIRECTLY from the problem statement — never pre-compute or simplify them, or (b) variable references like "$var_1.result$" using the exact output_parameter name from the referenced function's "output_parameters" definition.
- "label": a unique variable name (e.g., "$var_1", "$var_2") to capture this call's output for later reference.

Variable references follow the format: $<label>.<output_parameter_name>$

CRITICAL RULES for multi-step decomposition:
1. Each distinct arithmetic or logical operation MUST be its own separate function call — NEVER collapse multiple operations into one call or pre-compute intermediate values yourself.
2. Use the MINIMUM number of calls needed: avoid creating extra intermediate calls for constants that can be expressed as literal arguments.
3. When a sub-expression appears as an intermediate result, capture it with a labeled variable and reference it in subsequent calls using "$var_N.<output_parameter_name>$".
4. Argument values must be concrete numeric literals taken directly from the problem statement (integers or floats like 3, 5). NEVER use symbolic variable names (like "x", "y", "a") as argument values — these are not valid. Only use variable references ($var_N.result$) when the value comes from a prior function call's output.
5. If the same numeric sub-expression is needed in multiple places, create one call per occurrence with its own label.
6. When a problem contains a fraction like "3/5", you MUST compute it using a `divide` call: divide(arg_0=3, arg_1=5). Do NOT pass the fraction as a pre-computed decimal or collapse it into a multiply call. The fraction numerator and denominator are the literal arguments.
   EXCEPTION — CROSS-MULTIPLICATION: When the problem requires computing a ratio of totals (e.g., "what fraction of total boxes did crew A load?"), the canonical approach is to express numerator and denominator as products of integers using multiply calls, then divide(numerator_result, denominator_result). In this case, do NOT pre-divide the fractions — instead multiply out the common denominator form. Use the operation plan from STEP C to identify which approach yields the correct integer argument values.
7. Before writing any JSON, count the exact number of distinct operations needed. Your output array MUST contain exactly that many function call objects — no more, no fewer.
8. PERCENTAGES: NEVER convert a percentage to a decimal inline. Express "25%" as divide(25, 100), then use that result. WRONG: multiply(arg_0=80, arg_1=0.25). CORRECT: divide(arg_0=25, arg_1=100) → multiply(arg_0=$var_1.result$, arg_1=80).
9. SYMBOLIC VARIABLES: If the problem uses letters like x, y, z, a, w with no explicit numeric values, you MUST assign concrete test numbers satisfying all constraints, then derive the target expression's numeric value. Example: if "y is halfway between x and z", set x=4, z=0 → y=2, and use those integers. NEVER pass "x", "y", "z", "a", "w", or ANY letter as an argument value — this is always invalid, even in the first call. Violating this rule causes a score of 0 for the entire problem.
10. FINAL STEP COMPLETENESS: The LAST function call MUST produce the exact quantity the question asks for. If the question asks for a ratio of two computed values, the last call must be divide($var_A.result$, $var_B.result$). If it asks for a total combining two branches, the last call must combine them. NEVER stop one step before the answer.
11. INDEPENDENT BRANCHES: When two quantities are computed independently (parallel branches), each MUST have its own complete call sequence. Compute branch A fully, then branch B fully, then combine them in a final call.

EXAMPLES of correct vs. incorrect decomposition:

--- EXAMPLE A: Computing (3/5) * (2/5) ---
CORRECT — each fraction requires its own divide call, result referenced via variable:
[
  {"name": "divide", "label": "$var_1", "arguments": {"arg_0": 3, "arg_1": 5}},
  {"name": "divide", "label": "$var_2", "arguments": {"arg_0": 2, "arg_1": 5}},
  {"name": "multiply", "label": "$var_3", "arguments": {"arg_0": "$var_1.result$", "arg_1": "$var_2.result$"}}
]
WRONG — pre-computes the fractions as decimals, collapses 3 operations into 1:
[{"name": "multiply", "label": "$var_1", "arguments": {"arg_0": 0.6, "arg_1": 0.4}}]
WRONG — uses multiply instead of divide for the fraction computation:
[{"name": "multiply", "label": "$var_1", "arguments": {"arg_0": 3, "arg_1": 5}}, ...]

--- EXAMPLE B: Computing (36 + 32) - 59, then 36 - result ---
CORRECT — 3 operations = 3 calls, results chained via variable references:
[
  {"name": "add", "label": "$var_1", "arguments": {"arg_0": 36, "arg_1": 32}},
  {"name": "subtract", "label": "$var_2", "arguments": {"arg_0": "$var_1.result$", "arg_1": 59}},
  {"name": "subtract", "label": "$var_3", "arguments": {"arg_0": 36, "arg_1": "$var_2.result$"}}
]
WRONG — collapses 3 operations into 1, skips required intermediate steps:
[{"name": "subtract", "label": "$var_1", "arguments": {"arg_0": 59, "arg_1": 32}}]

--- EXAMPLE C: Problem with symbolic variables like "x", "y" ---
If the problem says "y is halfway between x and z" and asks you to compute a numeric answer using the relationships, you must derive the numeric values from the constraints and use ONLY concrete integers or floats as argument values.
WRONG — uses symbolic names as argument values (never valid):
[{"name": "subtract", "label": "$var_1", "arguments": {"arg_0": "x", "arg_1": "y"}}, ...]
CORRECT — uses the numeric values that result from solving the relationships algebraically:
[{"name": "add", "label": "$var_1", "arguments": {"arg_0": 2, "arg_1": 2}}, ...]

--- EXAMPLE D: Percentages — "20 liters added to 80 liters of a 25% mixture" ---
WRONG — converts 25% to decimal 0.25 inline:
[{"name": "multiply", "label": "$var_1", "arguments": {"arg_0": 80, "arg_1": 0.25}}, ...]
CORRECT — express 25% as divide(25, 100), then use the result:
[
  {"name": "divide", "label": "$var_1", "arguments": {"arg_0": 25, "arg_1": 100}},
  {"name": "multiply", "label": "$var_2", "arguments": {"arg_0": "$var_1.result$", "arg_1": 80}},
  {"name": "add", "label": "$var_3", "arguments": {"arg_0": 20, "arg_1": "$var_2.result$"}}
]

--- EXAMPLE E: Final-step completeness — "what is his speed in km per hour?" (distance=500m, time=4min) ---
Requires computing distance_km and time_hours independently, then dividing them.
WRONG — stops before the final ratio step (only 3 calls, missing the divide):
[{"name": "divide", "label": "$var_1", "arguments": {"arg_0": 500, "arg_1": 4}},
 {"name": "multiply", "label": "$var_2", "arguments": {"arg_0": "$var_1.result$", "arg_1": 60}},
 {"name": "divide", "label": "$var_3", "arguments": {"arg_0": "$var_2.result$", "arg_1": 1000}}]
CORRECT — two independent branches, then a final divide combining them:
[
  {"name": "divide", "label": "$var_1", "arguments": {"arg_0": 500, "arg_1": 1000}},
  {"name": "multiply", "label": "$var_2", "arguments": {"arg_0": 4, "arg_1": 60}},
  {"name": "divide", "label": "$var_3", "arguments": {"arg_0": "$var_2.result$", "arg_1": 3600}},
  {"name": "divide", "label": "$var_4", "arguments": {"arg_0": "$var_1.result$", "arg_1": "$var_3.result$"}}
]

--- EXAMPLE F: Sum-of-products — enumerate all multiplications before adding ---
"Compute (6 * 3) + (2 * 5) + (6 * 3)":
WRONG — collapses into fewer calls or wrong functions:
[{"name": "add", "label": "$var_1", "arguments": {"arg_0": 6, "arg_1": 4}}, ...]
CORRECT — every * is its own multiply call, every + is its own add call:
[
  {"name": "multiply", "label": "$var_1", "arguments": {"arg_0": 6, "arg_1": 3}},
  {"name": "multiply", "label": "$var_2", "arguments": {"arg_0": 2, "arg_1": 5}},
  {"name": "multiply", "label": "$var_3", "arguments": {"arg_0": 6, "arg_1": 3}},
  {"name": "add", "label": "$var_4", "arguments": {"arg_0": "$var_1.result$", "arg_1": "$var_2.result$"}},
  {"name": "add", "label": "$var_5", "arguments": {"arg_0": "$var_4.result$", "arg_1": "$var_3.result$"}}
]

You MUST NOT include any other text in the response besides the JSON list. Before generating the JSON, you MUST complete a mandatory planning phase (silently, without outputting it). Execute these planning steps in order:
STEP A — SOLVE ALGEBRAICALLY FIRST: If the problem contains symbolic variables (x, y, z, a, b, w) or word-problem unknowns with no explicit numeric values, assign concrete test numbers that satisfy all stated relationships (e.g., if "y is halfway between x and z", let x=4, y=2, z=0 or derive the ratio). Compute the target expression numerically. Use ONLY those concrete integers or floats in all arguments. NEVER output a letter as an argument value.

STEP B — IDENTIFY THE FINAL EXPRESSION: Write out the complete mathematical expression the question asks for (e.g., "day_crew_boxes / total_boxes" or "(36+32−59) subtracted from 36"). Every sub-expression in this final expression becomes a function call.

STEP C — BUILD AN ORDERED OPERATION LIST: Number every distinct primitive operation needed, starting from the leaf-level computations (those using only problem literals) up to the root (final answer). Example:
  Op1: multiply(10, 25) → $var_1   [leaf: literals only]
  Op2: multiply(15, $var_1.result$) → $var_2   [depends on Op1]
  Op3: divide($var_1.result$, $var_2.result$) → $var_3   [final]
The JSON array must have exactly one object per operation in this list, in the same order. Leaf operations (no variable references) MUST come before any operation that consumes their results.

STEP D — VERIFY ARGUMENT VALUES: For each operation in your list, confirm every argument is either (a) a concrete integer/float literal copied directly from the problem, or (b) a $var_N.result$ reference where N is the label of a prior operation. If any argument is a letter, a decimal approximation of a fraction, or a pre-computed combination of problem values, STOP and fix it.

STEP E — COUNT AND CONFIRM: Count the operations in your list. Your JSON array MUST contain exactly that many objects. The LAST object must produce the exact quantity the question asks for.

(6) Only after completing Steps A–E, write the final JSON array.

Here is a list of available functions in JSON format:
{{tools}}

Question: {{user_question}}
```
</details>

**A note on polling.** `utils.run_job(...)` orchestrates upload → submit → poll → download. We pass it an `on_status` callback that overwrites a single Markdown line in place — no scrolling log spam over the 20–50 minute job run.

If `APO_USE_REFERENCE=1` is set, `run_job` returns the bundled `reference_results.jsonl` without calling AWS.

---
## Chapter B — PDF (MMVQA)

The multimodal variant: the model is shown a scientific PDF as an attachment and asked a question about its content. The metric Lambda computes **ROUGE-L F1** between the answer and the gold passage.

The only structural change is **`inputVariablesMultimodal`** on each sample, pointing at an `s3://` URI for the PDF.

### What is ROUGE-L F1?

A text-overlap metric that scores predictions by their **longest common subsequence (LCS)** with the reference. Tokens in the LCS must appear in the same order in both texts but do not need to be contiguous, so paraphrases and reorderings cost less than they would under exact match.

| Quantity | Formula |
|---|---|
| LCS length | longest in-order token overlap between prediction and reference |
| Precision (P) | LCS / `len(prediction tokens)` |
| Recall (R) | LCS / `len(reference tokens)` |
| **ROUGE-L F1** | `2 · P · R / (P + R)`, range `[0.0, 1.0]` |

**Tiny worked example**
- Prediction: `the cat sat on the mat`
- Reference: `a cat is on the mat`
- LCS: `cat on the mat` → length 4
- P = 4 / 6 ≈ 0.667 &nbsp;&nbsp; R = 4 / 6 ≈ 0.667 &nbsp;&nbsp; F1 ≈ **0.667**

**Why this metric for MMVQA**
- The gold answers are short free-text passages, not single tokens — exact match would be too brittle.
- Rewards keeping the right key tokens (entities, numbers) in roughly the right order.
- Deterministic, language-agnostic, fast — perfect fit for the Lambda allowlist (the bundled `lambda_function.py` implements it in pure stdlib `re` + `collections`).


In [6]:
# Upload PDFs once (idempotent — ETag check skips already-synced files).
if not USING_REPLAY:
    utils.sync_assets("mmvqa", env, s3=s3)

# Preview the record shape with multimodal entries attached.
preview_mm = utils.build_lambda_record(
    "mmvqa",
    lambda_arn="arn:aws:lambda:us-west-2:000000000000:function:placeholder",
    bucket=env["BUCKET"],
)
display(utils.render_dimensions(preview_mm))
display(Markdown("**First sample shape (note `inputVariablesMultimodal`):**"))
display(utils.render_sample_shape(preview_mm))

| Field | Value (truncated) |
|---|---|
| `inputVariables[0].question` | What are the limitations of the study in understanding the population structure of C. hominis based on MLST? |
| `inputVariablesMultimodal[0].document_path` | type=`pdf` s3Uri=`s3://apo-tutorial-686255965275-us-west-2/apo-tutorial/assets/mmvqa/PMC5460555.pdf` |
| `referenceResponse` | In  conclusion,  our  study  demonstrated  population structure of C. hominis by MLST typing.  the  population structure of C. hominis by MLST typing.  MLST is a better tool to study transmission dynamics  of Cryptosporidium subtypes due to … |

> **Heads up — long-running cell.** The call below submits a real APO job that can take up to ~20 minutes per chapter. The cell will keep its `status` / `elapsed` display refreshing until the job reaches a terminal state (`Completed`, `Failed`, or `Stopped`). Safe to leave running while you work on something else.

In [ ]:
# Bump the target model up to one with PDF understanding.
TARGET_MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"

mmvqa_arn = utils.lambda_arn_for(env, "mmvqa") if not USING_REPLAY else "arn:aws:lambda:us-west-2:replay:function:mmvqa"
record = utils.build_lambda_record("mmvqa", lambda_arn=mmvqa_arn, bucket=env["BUCKET"])
display(utils.render_record_shape(record))

handle = display(Markdown("_waiting for status…_"), display_id=True)
def on_status(status, elapsed_s):
    handle.update(Markdown(f"**status:** `{status}` &nbsp;|&nbsp; **elapsed:** `{elapsed_s:.0f}s`"))

mmvqa_results = utils.run_job(
    record,
    example="mmvqa",
    model_id=TARGET_MODEL_ID,
    env=env,
    on_status=on_status,
)
print(f"results file: {mmvqa_results}")


In [ ]:
parsed = utils.parse_results(mmvqa_results)
display(utils.render_results_table(parsed))
for row in parsed:
    display(utils.render_prompt_diff(row))

---
## Recap

You deployed two metric Lambdas, attached them to APO records, submitted jobs through `boto3`, and rendered original-vs-optimized scores.

**Next:** `02_llm_as_judge.ipynb` — same flow but the metric is a judge model with a natural-language rubric, no Lambda required.